<a id="06"></a>

## 06 — Window Leakage Analysis
### Mục 0.3 (phần 1) — Rò rỉ dữ liệu do Sliding Window

**Mục đích**: minh họa bằng số **bản chất** của rò rỉ dữ liệu khi chia
ngẫu nhiên theo sliding window, và chạy thực nghiệm nhỏ đối chiếu Random
Window Split vs File-based Split — bằng chứng thực nghiệm **sơ bộ** cho
RQ1/H1 (kết quả đầy đủ, chính thức vẫn báo cáo lại ở Giai đoạn 4 với bể
đặc trưng đầy đủ và LOLO 4-fold).

**Lưu ý phương pháp quan trọng**: minh họa dưới đây đo **tỉ lệ mẫu thô
thực sự trùng nhau** giữa 2 window liền kề (một sự thật xác định) và
**chênh lệch đặc trưng thống kê** giữa chúng — KHÔNG dùng hệ số tương quan
Pearson so theo vị trí chỉ số (`corrcoef(w1, w2)`), vì cách đó có thể cho
số liệu sai lệch/gây hiểu nhầm khi tín hiệu có thành phần tuần hoàn lệch
pha ngẫu nhiên với bước dịch cửa sổ.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display
from typing import Any
from matplotlib.patches import Rectangle

from common import io_utils, pipeline, synthetic, dsp, features, splitting, config as cfg

pd.set_option("display.max_colwidth", 120)

In [2]:
USE_SYNTHETIC_DATA = False
REAL_DATA_ROOT = Path("../../data/raw")            # <-- data/raw/
SYNTHETIC_DATA_ROOT = Path("./_data/synthetic_cwru")
OUTPUT_DIR = Path("./outputs")
FORCE_REBUILD_MANIFEST = False  # manifest đã được build ở notebook 01

FIGURES_DIR = OUTPUT_DIR / "figures"
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
manifest = pipeline.get_manifest(
    use_synthetic=USE_SYNTHETIC_DATA,
    real_data_root=REAL_DATA_ROOT,
    synthetic_data_root=SYNTHETIC_DATA_ROOT,
    output_dir=OUTPUT_DIR,
    force_rebuild=FORCE_REBUILD_MANIFEST,
)
print(f"Tổng số file trong manifest: {len(manifest)}")
manifest.head()

Tổng số file trong manifest: 161


,file_path,load_hp,label,fault_diameter_mils,or_position,source_category,sensor_location,declared_sample_rate_khz,n_samples_DE,n_samples_FE,n_samples_BA,rpm_from_file,read_error,warnings,has_warning,resolved_sample_rate_hz
0,..\..\data\raw\12k_Drive_End_Bearing_Fault_Data\B\007\118_0.mat,0,B,7.0,NaN,12k_Drive_End_Bearing_Fault_Data,DE,12.0,122571,122571.0,122571.0,1796.0,NaN,NaN,False,12000.0
1,..\..\data\raw\12k_Drive_End_Bearing_Fault_Data\B\007\119_1.mat,1,B,7.0,NaN,12k_Drive_End_Bearing_Fault_Data,DE,12.0,121410,121410.0,121410.0,1772.0,NaN,NaN,False,12000.0
2,..\..\data\raw\12k_Drive_End_Bearing_Fault_Data\B\007\120_2.mat,2,B,7.0,NaN,12k_Drive_End_Bearing_Fault_Data,DE,12.0,121556,121556.0,121556.0,1748.0,NaN,NaN,False,12000.0
3,..\..\data\raw\12k_Drive_End_Bearing_Fault_Data\B\007\121_3.mat,3,B,7.0,NaN,12k_Drive_End_Bearing_Fault_Data,DE,12.0,121556,121556.0,121556.0,1722.0,NaN,NaN,False,12000.0
4,..\..\data\raw\12k_Drive_End_Bearing_Fault_Data\B\014\185_0.mat,0,B,14.0,NaN,12k_Drive_End_Bearing_Fault_Data,DE,12.0,121846,121846.0,121846.0,1796.0,NaN,NaN,False,12000.0


## Lọc phạm vi dữ liệu (mục 0.1/0.4) — TRƯỚC khi phân tích

`manifest` gốc còn chứa cả file NGOÀI phạm vi đã chốt (vòng bi NTN đường
kính 28/40 mils, vị trí Outer Race Orthogonal/Opposite). Nếu không lọc ở
đây, các bước chọn "file đại diện" bên dưới có thể vô tình chọn nhầm — vd
chọn file OR vị trí Orthogonal/Opposite trong khi đề tài chỉ dùng Centered.

In [4]:
manifest = io_utils.apply_scope_filter(manifest)

Áp dụng bộ lọc phạm vi (mục 0.1/0.4):
  - Loại 8 file có cảnh báo 'VONG_BI_NTN'
  - Loại 47 file có cảnh báo 'OR_NGOAI_PHAM_VI'
  - Loại 0 file có cảnh báo 'OR_THIEU_VI_TRI'
  - Loại 0 file có cảnh báo 'THIEU_NHAN'
  - Loại 0 file có cảnh báo 'THIEU_TAI'
  - Loại 0 file có cảnh báo 'DUONG_KINH_LA'
  - Loại 45 file có cảnh báo 'NGOAI_PHAM_VI_CAM_BIEN'
  - Loại 52 file có cảnh báo 'NGOAI_PHAM_VI_TAN_SO_KHAI_BAO'
  - Loại 0 file có cảnh báo 'THIEU_TIN_HIEU_DE'
  - Loại 0 file có cảnh báo 'LOI_DOC_FILE'
Tổng: loại 121/161 file — còn lại 40 file.


## Minh họa 1 — Tỉ lệ mẫu trùng & chênh lệch đặc trưng giữa 2 window liền kề

In [5]:
def demo_window_overlap_similarity(signal, window_size, overlap_ratio):
    step = max(int(window_size * (1 - overlap_ratio)), 1)
    w1 = signal[0: window_size]
    w2 = signal[step: step + window_size]

    shared_fraction = max(0, window_size - step) / window_size
    f1 = features.extract_simple_features(w1)
    f2 = features.extract_simple_features(w2)

    print(f"Overlap cấu hình = {overlap_ratio*100:.0f}%  (step={step}, window={window_size})")
    print(f"  -> Tỉ lệ MẪU THÔ trùng nhau: {shared_fraction*100:.1f}%")
    print("  -> Chênh lệch đặc trưng thống kê giữa 2 window:")
    for name in f1:
        rel_diff = abs(f1[name] - f2[name]) / (abs(f1[name]) + 1e-8) * 100
        print(f"       {name:10s}: {rel_diff:6.2f}% chênh lệch")
    print()
    return shared_fraction

fp = pipeline.pick_file(manifest, label="OR", load_hp=0)
demo_signal = io_utils.load_de_signal(Path(fp))

for overlap in (0.75, 0.5, 0.0):
    demo_window_overlap_similarity(demo_signal, window_size=2048, overlap_ratio=overlap)

Overlap cấu hình = 75%  (step=512, window=2048)
  -> Tỉ lệ MẪU THÔ trùng nhau: 75.0%
  -> Chênh lệch đặc trưng thống kê giữa 2 window:
       rms       :   1.55% chênh lệch
       kurtosis  :   3.37% chênh lệch
       skewness  :  24.15% chênh lệch
       peak      :   0.00% chênh lệch
       std       :   1.56% chênh lệch

Overlap cấu hình = 50%  (step=1024, window=2048)
  -> Tỉ lệ MẪU THÔ trùng nhau: 50.0%
  -> Chênh lệch đặc trưng thống kê giữa 2 window:
       rms       :   0.17% chênh lệch
       kurtosis  :   0.15% chênh lệch
       skewness  :  14.44% chênh lệch
       peak      :   0.00% chênh lệch
       std       :   0.17% chênh lệch

Overlap cấu hình = 0%  (step=2048, window=2048)
  -> Tỉ lệ MẪU THÔ trùng nhau: 0.0%
  -> Chênh lệch đặc trưng thống kê giữa 2 window:
       rms       :   2.50% chênh lệch
       kurtosis  :   1.03% chênh lệch
       skewness  :  63.24% chênh lệch
       peak      :   6.27% chênh lệch
       std       :   2.51% chênh lệch



## Minh họa 2 — Thực nghiệm so sánh Random Window Split vs File-based Split

Cắt sliding window (overlap 50%) trên **toàn bộ** file trong manifest,
trích 5 đặc trưng đơn giản, huấn luyện Random Forest theo 2 cách chia,
so sánh accuracy.

In [6]:
WINDOW_SIZE = 2048
OVERLAP_RATIO = 0.5

all_windows = []
for _, row in manifest.iterrows():
    if row["label"] is None or pd.isna(row["label"]):
        continue
    x = io_utils.load_de_signal(Path(row["file_path"]))
    file_id = f"{row['label']}_{row['load_hp']}_{row.get('fault_diameter_mils')}_{row['file_path']}"
    w = features.make_sliding_windows(x, WINDOW_SIZE, OVERLAP_RATIO, file_id=file_id, label=row["label"])
    all_windows.append(w)

windows_df = pd.concat(all_windows, ignore_index=True)
feature_df = features.build_feature_table(windows_df)
feature_cols = ["rms", "kurtosis", "skewness", "peak", "std"]
print(f"Tổng số window: {len(feature_df)}  (từ {feature_df['file_id'].nunique()} file)")

Tổng số window: 5886  (từ 40 file)


In [7]:
result_df = splitting.run_split_comparison_experiment(feature_df, feature_cols)

# Xem cấu trúc kết quả
print("Các cột hiện có:", result_df.columns.tolist())
display(result_df)

Các cột hiện có: ['split_method', 'accuracy', 'f1_macro', 'n_train', 'n_test']


,split_method,accuracy,f1_macro,n_train,n_test
0,Random Window Split,0.986402,0.985963,4121.0,1765.0
1,File-based Split,0.880042,0.879538,4002.0,1884.0


In [8]:
# Dùng result_df đã có từ cell trên — KHÔNG gọi lại experiment
result_df.to_csv(TABLES_DIR / "06_split_comparison.csv")

# Lấy accuracy ở dòng 0 (Random Split) và dòng 1 (File-based Split)
acc_random = float(result_df["accuracy"].iloc[0])
acc_file   = float(result_df["accuracy"].iloc[1])
delta = acc_random - acc_file

print(result_df)
print(f"\nΔAccuracy (ảo phồng do window leakage) = {delta:.4f}")
print(f"→ Random Split: {acc_random:.4f}, File-based: {acc_file:.4f}")

          split_method  accuracy  f1_macro  n_train  n_test
0  Random Window Split  0.986402  0.985963   4121.0  1765.0
1     File-based Split  0.880042  0.879538   4002.0  1884.0

ΔAccuracy (ảo phồng do window leakage) = 0.1064
→ Random Split: 0.9864, File-based: 0.8800


## Quan sát & kết luận sơ bộ

- Ghi lại ΔAccuracy quan sát được — đây là bằng chứng **sơ bộ** cho RQ1/H1.
- Kết quả **chính thức** để đưa vào Bảng 1 của báo cáo (mục 4.1) phải chạy
  lại ở Giai đoạn 4 với: (a) bể đặc trưng đầy đủ (mục 1.4, không chỉ 5 đặc
  trưng đơn giản ở đây), và (b) File-based Split + LOLO đầy đủ 4-fold
  (không chỉ 1 lần chia ngẫu nhiên như demo này).

> ⚠️ **Vì sao ΔAccuracy có thể RẤT NHỎ khi chạy với `USE_SYNTHETIC_DATA =
> True`?** Tín hiệu giả lập trong `common/synthetic.py` tạo ra các lớp
> tách biệt rất rõ ràng (mỗi lỗi có 1 tần số xung riêng biệt, không nhiễu
> chồng lấp giữa các lớp) — cả 2 cách chia đều dễ dàng đạt gần 100%
> accuracy ("hiệu ứng trần"/ceiling effect), khiến chênh lệch do rò rỉ dữ
> liệu bị che khuất dù nó vẫn tồn tại về bản chất (xem lại Minh họa 1 ở
> trên — tỉ lệ mẫu trùng và chênh lệch đặc trưng vẫn rất rõ). **Đây KHÔNG
> phải bằng chứng phản bác H1.** Với dữ liệu CWRU thật (nhiễu cơ khí thật,
> ranh giới giữa các lớp mờ hơn nhiều), y văn báo cáo chênh lệch do rò rỉ
> thường rất đáng kể (accuracy ảo >99% so với accuracy tổng quát hóa thấp
> hơn nhiều) — đây chính là động lực ban đầu của đề tài (xem mục 0 —
> Bối cảnh và động lực nghiên cứu). Chạy lại notebook này với dữ liệu thật
> để có con số đáng tin cậy.

Bảng kết quả đã lưu tại `outputs/tables/06_split_comparison.csv`.